---
title: Rolling volatility
jupyter: python3
execute:
  enabled: true
---

`q.indicator.rolling_volatility` calculates the rolling standard deviation of a return `Series`. It does not annualize the result, so its units match the sampling frequency of the input returns.

By default, each value requires a complete `window`. Set `min_periods` explicitly when a shorter warm-up is appropriate.

In [1]:
import numpy as np
import pandas as pd

import qrt as q

prices = pd.concat(
    {"AAPL": q.data.datasets.load("aapl")["close"], "SPY": q.data.datasets.load("spy")["close"]},
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
daily_returns = prices.apply(q.indicator.returns)

## Annualize daily volatility

For daily data, multiplying by $\sqrt{252}$ expresses the estimate on an approximate annual scale. This scaling remains separate because the correct factor depends on input frequency.

In [2]:
window = 21
annualized_volatility = daily_returns.apply(
    lambda series: q.indicator.rolling_volatility(series, window=window)
).mul(np.sqrt(252) * 100)
annualized_volatility.tail()

,AAPL,SPY
datetime,,
2026-07-17,35.868887,12.729275
2026-07-20,36.645054,11.950500
2026-07-21,36.640734,11.749494
2026-07-22,36.713390,11.699889
2026-07-23,36.919826,11.374673


In [3]:
figure = q.plot.col(
    annualized_volatility.add_suffix(f" {window}-session volatility"),
    title="AAPL and SPY rolling annualized volatility",
    ylabel="Annualized volatility (%)",
)
figure.show(renderer="notebook_connected")